In [106]:
#Load the environment variables
from dotenv import load_dotenv
load_dotenv()

True

In [107]:
### LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-safeguard-20b"
)

llm.invoke("What is Langchain? in a single sentence").content.strip()

'LangChain is an open‑source framework that streamlines building applications powered by large language models by providing modular components for prompt construction, memory, and integration with external tools and APIs.'

#### **Load the Hospital Documents**

In [108]:
## Validate the path
import os
path ="../../Hospital_RAG_documents"

if os.path.exists(path):
    print("Path is Valid")
else:
    print("Path is not Valid")


Path is Valid


In [109]:
### Document Loader
from langchain_community.document_loaders import DirectoryLoader,TextLoader

loader = DirectoryLoader(
    path,
    glob = "*.md",
    loader_cls = TextLoader,
    loader_kwargs={"encoding":"utf-8"}
)

documents = loader.load()

print("Number Of Documents:",len(documents))

Number Of Documents: 4


#### **Split the documents using Semantic Chunking**

In [110]:
### Embedding Model
### Create a Embedding model by HuggingFace with 1024 dimensions
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
        model = "BAAI/bge-large-en-v1.5",
        model_kwargs = {"device":"cpu"},
        encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

### ***Semantic Chunking means split the data based on semantic meaning instead of chunk_size***

In [111]:
from langchain_experimental.text_splitter import SemanticChunker

splitter= SemanticChunker(
    breakpoint_threshold_type = "percentile",
    breakpoint_threshold_amount = 90,
    embeddings=embedding_model
)

#### **Rechunking**

In [112]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 400,
    chunk_overlap = 60
)

In [113]:
chunks = recursive_splitter.split_documents(documents)
chunks

[Document(metadata={'source': '..\\..\\Hospital_RAG_documents\\departments.md'}, page_content='# ABC General Hospital — Departments\n\n## Department Overview\n\nABC General Hospital has several clinical departments.'),
 Document(metadata={'source': '..\\..\\Hospital_RAG_documents\\departments.md'}, page_content='| Department | Main Area | Typical Services |\n|---|---|---|\n| Cardiology | Heart and cardiovascular system | Cardiac evaluation and cardiovascular care |\n| Neurology | Brain and nervous system | Neurological evaluation and care |\n| Orthopedics | Bones, joints, and muscles | Musculoskeletal evaluation and treatment |\n| Pediatrics | Children and adolescents | Pediatric consultations and care |'),
 Document(metadata={'source': '..\\..\\Hospital_RAG_documents\\departments.md'}, page_content='| Dermatology | Skin, hair, and nails | Dermatological evaluation and care |\n| General Medicine | General adult healthcare | General medical evaluation and management |'),
 Document(metad

In [114]:
### Number of chunks
len(chunks)

20

#### ***Store in Chroma***

In [115]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    chunks,
    embedding=embedding_model,
    persist_directory="../../vectorstore/hospital_rag",
    collection_name="hospital_rag"
)

In [116]:
query = "What services are available in the Cardiology Department?"

In [117]:
query_vector = embedding_model.embed_query(query)
print(len(query_vector))

1024


#### **Create and test the retriever**

In [182]:
vector_retriever = vectorstore.as_retriever(search_type = "similarity",search_kwargs = {"k":5})
retrieved_docs = vector_retriever.invoke(query)
retrieved_docs

[Document(id='ae9fcc8c-be56-45c8-8963-19ee749211d9', metadata={'source': '..\\..\\Hospital_RAG_documents\\visiting_hours.md'}, page_content='## Important Note\n\nThese visiting hours are fictional and created for RAG experimentation. They are not the schedule of a real hospital.'),
 Document(id='2c226fd9-2135-44bf-8f4e-c2d980530b43', metadata={'source': '..\\..\\Hospital_RAG_documents\\visiting_hours.md'}, page_content='## Important Note\n\nThese visiting hours are fictional and created for RAG experimentation. They are not the schedule of a real hospital.'),
 Document(id='59235a2c-4313-4a8e-8938-5b65941825d9', metadata={'source': '..\\..\\Hospital_RAG_documents\\departments.md'}, page_content='## Important Note\n\nThis is fictional hospital information created for a LangChain RAG project. It is not clinical guidance.'),
 Document(id='4c9ed752-23f1-4769-9bbf-2641d5d89c25', metadata={'source': '..\\..\\Hospital_RAG_documents\\departments.md'}, page_content='## Important Note\n\nThis is 

In [183]:
def format_docs(documents):

    return "\n\n".join(doc.page_content for doc in documents)

In [184]:
# design a prompt
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
You are a Kubernetes documentation Assistant.

Answer the question using only the provided Context only.


Rules:
    1.Don't use information outside the Context.
    2.If the answer is not available in the context,
    say I don't have information based on the Provided Documents.

Context:
{context}

Question:
{question}

Answer:
""")

#### ***Step 6 — Build the RAG flow***

In [185]:
### Vector similarity search
def rag(query):

    retrieved_docs  =  vector_retriever.invoke(query)
    

    context = format_docs(retrieved_docs)

    message = prompt.invoke({"context":context,"question":query})

    response = llm.invoke(message).content

    return response


In [122]:
answer = rag(query)
print(answer)

Cardiology offers cardiac evaluation and cardiovascular care.


#### ***BM25 Retriver***

In [123]:
from langchain_classic.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(chunks)

In [186]:
bm25_retriever.k=5

#### ***Hybrid Search = Vector + BM25 Retriever***

In [187]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever,bm25_retriever],
    weights= [0.7,0.3]
)



In [188]:

def hybrid_rag(query):

    retrieved_docs  =  hybrid_retriever.invoke(query)
    

    context = format_docs(retrieved_docs)

    message = prompt.invoke({"context":context,"question":query})

    response = llm.invoke(message).content

    return response


In [189]:
answer = rag(query)
print(answer)

I don't have information based on the Provided Documents.


In [191]:
print("What are the visiting hours for the ICU?")
vector_response = rag("What are the visiting hours for the ICU?")
hybrid_response = hybrid_rag("What are the visiting hours for the ICU?")
print("Vector Response:",vector_response)
print("Hybrid Response:",hybrid_response)

What are the visiting hours for the ICU?
Vector Response: The ICU visiting hours are **2:00 PM to 3:00 PM, Monday through Sunday**.
Hybrid Response: The ICU visiting hours are **2:00 PM to 3:00 PM, Monday through Sunday**.


#### ***Step 7 — Add the Patient-Data Tool***

In [192]:
import pandas as pd

In [130]:
patients_df = pd.read_csv("../../Data/patients.csv")
patients_df[:5]

,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,MIDDLE,LAST,...,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME
0,cc3eac4a-34a7-9a15-1522-8087135631b7,1978-01-16,NaN,999-55-9547,S99974479,X81842955X,Mr.,Guillermo498,NaN,VonRueden376,...,North Brookfield,Massachusetts,Worcester County,25027.0,1535,42.267480,-72.105242,104532.20,18048.65,42283
1,8b9df7af-24b3-66ee-afeb-2734ebcbd33c,2001-02-11,NaN,999-32-1995,S99911042,X98255942X,Ms.,Jayne73,Mira443,Bruen238,...,Bedford,Massachusetts,Middlesex County,25017.0,1730,42.539821,-71.256841,98009.84,531508.43,33657
2,65cbad6a-eb9d-420e-9bab-b4f9c6aab7db,1948-06-15,NaN,999-23-9791,S99928100,X83081075X,Mrs.,Grisel924,Dortha70,Balistreri607,...,Quincy,Massachusetts,Norfolk County,25021.0,2170,42.204748,-70.988341,796732.91,219824.56,64779
3,9a3814bb-f5b0-05f3-928e-fe6640d5aa2a,2015-07-19,NaN,999-56-4831,NaN,NaN,NaN,Esperanza675,Rachell479,Auer97,...,Taunton,Massachusetts,Bristol County,25005.0,2718,41.902707,-70.976300,44050.63,17747.77,34422
4,6a703775-432b-ba7d-d53a-c4d7bb0f5b58,1995-01-23,NaN,999-91-2539,S99931120,X52632004X,Mr.,Arnulfo253,NaN,Boyer713,...,Boston,Massachusetts,Suffolk County,25025.0,2116,42.359644,-71.079948,5450.00,97459.94,6565


In [131]:
patients_df.columns

Index(['Id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX',
       'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE',
       'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE',
       'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES',
       'HEALTHCARE_COVERAGE', 'INCOME'],
      dtype='str')

In [132]:
patients_df.head()

,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,MIDDLE,LAST,...,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME
0,cc3eac4a-34a7-9a15-1522-8087135631b7,1978-01-16,NaN,999-55-9547,S99974479,X81842955X,Mr.,Guillermo498,NaN,VonRueden376,...,North Brookfield,Massachusetts,Worcester County,25027.0,1535,42.267480,-72.105242,104532.20,18048.65,42283
1,8b9df7af-24b3-66ee-afeb-2734ebcbd33c,2001-02-11,NaN,999-32-1995,S99911042,X98255942X,Ms.,Jayne73,Mira443,Bruen238,...,Bedford,Massachusetts,Middlesex County,25017.0,1730,42.539821,-71.256841,98009.84,531508.43,33657
2,65cbad6a-eb9d-420e-9bab-b4f9c6aab7db,1948-06-15,NaN,999-23-9791,S99928100,X83081075X,Mrs.,Grisel924,Dortha70,Balistreri607,...,Quincy,Massachusetts,Norfolk County,25021.0,2170,42.204748,-70.988341,796732.91,219824.56,64779
3,9a3814bb-f5b0-05f3-928e-fe6640d5aa2a,2015-07-19,NaN,999-56-4831,NaN,NaN,NaN,Esperanza675,Rachell479,Auer97,...,Taunton,Massachusetts,Bristol County,25005.0,2718,41.902707,-70.976300,44050.63,17747.77,34422
4,6a703775-432b-ba7d-d53a-c4d7bb0f5b58,1995-01-23,NaN,999-91-2539,S99931120,X52632004X,Mr.,Arnulfo253,NaN,Boyer713,...,Boston,Massachusetts,Suffolk County,25025.0,2116,42.359644,-71.079948,5450.00,97459.94,6565


In [133]:
patients_df.shape

(108, 28)

In [134]:
patient_columns = [
    "Id",
    "PREFIX",
    "FIRST",
    "MIDDLE",
    "LAST",
    "SUFFIX",
    "BIRTHDATE",
    "GENDER",
    "MARITAL",
    "BIRTHPLACE",
    "CITY",
    "STATE",
    "ZIP",
    "HEALTHCARE_EXPENSES",
    "HEALTHCARE_COVERAGE",
    "INCOME"
]

patients_df = patients_df[patient_columns].copy()

In [135]:
patients_df['Id'][:5]

0    cc3eac4a-34a7-9a15-1522-8087135631b7
1    8b9df7af-24b3-66ee-afeb-2734ebcbd33c
2    65cbad6a-eb9d-420e-9bab-b4f9c6aab7db
3    9a3814bb-f5b0-05f3-928e-fe6640d5aa2a
4    6a703775-432b-ba7d-d53a-c4d7bb0f5b58
Name: Id, dtype: str

In [136]:
name_columns = ["PREFIX", "FIRST", "MIDDLE", "LAST", "SUFFIX"]

patients_df["Name"] = (
    patients_df[name_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [137]:
patients_df = patients_df.drop(columns=name_columns)

In [193]:
from langchain_core.tools import tool

@tool
def search_patient(patient_id):

    """Search the hospital's synthetic patient records using a patient ID
     and return the matching patient information."""

    patient = patients_df[patients_df["Id"]==patient_id]

    if patient.empty:
        return "Patient not found"

    return patient.to_dict(orient = "records")[0]

In [194]:
search_patient.invoke("9a3814bb-f5b0-05f3-928e-fe6640d5aa2a")

{'Id': '9a3814bb-f5b0-05f3-928e-fe6640d5aa2a',
 'BIRTHDATE': '2015-07-19',
 'GENDER': 'F',
 'MARITAL': nan,
 'BIRTHPLACE': 'Stoughton  Massachusetts  US',
 'CITY': 'Taunton',
 'STATE': 'Massachusetts',
 'ZIP': 2718,
 'HEALTHCARE_EXPENSES': 44050.63,
 'HEALTHCARE_COVERAGE': 17747.77,
 'INCOME': 34422,
 'Name': 'Esperanza675 Rachell479 Auer97'}

#### **Reranking**

In [195]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [201]:
@tool
def hospital_search(query: str):
    """Search the hospital knowledge base and answer questions using information retrieved from the hospital documents."""
    
    retrieved_docs = hybrid_retriever.invoke(query)

    pairs = [[query,doc.page_content] for doc in retrieved_docs]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(zip(retrieved_docs,scores), key = lambda x:x[1],reverse=True)

    top_docs = [doc for doc,score in ranked_docs[:3]]



    context = format_docs(top_docs)

    message = prompt.invoke({
        "context": context,
        "question": query
    })

    response = llm.invoke(message).content

    return response

In [141]:
hospital_search.invoke({
    "query": "What are the ICU visiting hours?"
})


--- Document 1 ---
## General Ward

Visitors may visit patients in the General Ward during the scheduled visiting periods.

## Intensive Care Unit (ICU)

ICU visiting is restricted to the scheduled period. Visitors should follow instructions provided by hospital staff.

## Pediatrics

Pediatric department visiting hours are scheduled from 4:00 PM to 6:00 PM, Monday through Saturday.

## Outpatient Department

--- Document 2 ---
# ABC General Hospital — Visiting Hours

## General Visiting Schedule

| Area | Visiting Hours | Days |
|---|---|---|
| General Ward | 10:00 AM - 12:00 PM | Monday - Sunday |
| General Ward | 4:00 PM - 6:00 PM | Monday - Sunday |
| ICU | 2:00 PM - 3:00 PM | Monday - Sunday |
| Pediatrics | 4:00 PM - 6:00 PM | Monday - Saturday |
| Outpatient Department | 9:00 AM - 1:00 PM | Monday - Saturday |

## General Ward

Visitors may visit patients in the General Ward during the scheduled visiting periods. ## Intensive Care Unit (ICU)

ICU visiting is restricted to the s

'ICU visiting hours are 2:00\u202fPM to 3:00\u202fPM, Monday through Sunday.'

In [142]:
hospital_search.invoke({
    "query": "What services are available in the Cardiology Department?"
})


--- Document 1 ---
## Cardiology

The Cardiology Department focuses on evaluation and management of cardiovascular conditions.

## Neurology

The Neurology Department focuses on conditions involving the brain, spinal cord, nerves, and related neurological functions.

## Orthopedics

The Orthopedics Department focuses on musculoskeletal conditions involving bones, joints, muscles, ligaments, and related structures.

--- Document 2 ---
| Department | Main Area | Typical Services |
|---|---|---|
| Cardiology | Heart and cardiovascular system | Cardiac evaluation and cardiovascular care |
| Neurology | Brain and nervous system | Neurological evaluation and care |
| Orthopedics | Bones, joints, and muscles | Musculoskeletal evaluation and treatment |
| Pediatrics | Children and adolescents | Pediatric consultations and care |

--- Document 3 ---
Patients with urgent medical conditions should report to the Emergency Department. Emergency cases are prioritized according to clinical urgency.


'Cardiology provides cardiac evaluation and cardiovascular care, focusing on the evaluation and management of cardiovascular conditions.'

In [202]:
tools = [
    hospital_search,
    search_patient
]

In [203]:
llm_with_tools = llm.bind_tools(tools)

In [204]:
tool_map = {tool.name:tool for tool in tools}
tool_map

{'hospital_search': StructuredTool(name='hospital_search', description='Search the hospital knowledge base and answer questions using information retrieved from the hospital documents.', args_schema=<class 'langchain_core.utils.pydantic.hospital_search'>, func=<function hospital_search at 0x000001D2F1180400>),
 'search_patient': StructuredTool(name='search_patient', description="Search the hospital's synthetic patient records using a patient ID\n     and return the matching patient information.", args_schema=<class 'langchain_core.utils.pydantic.search_patient'>, func=<function search_patient at 0x000001D2F116AF20>)}

In [205]:
llm_with_tools.invoke("What are the ICU visiting hours?")

AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks about ICU visiting hours. We need to search hospital knowledge base. Use hospital_search with query "ICU visiting hours".', 'tool_calls': [{'id': 'fc_06e32047-8111-4f9d-828a-e40dc9b4a887', 'function': {'arguments': '{"query":"ICU visiting hours"}', 'name': 'hospital_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 173, 'total_tokens': 227, 'completion_time': 0.05439957, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.009692284, 'prompt_tokens_details': None, 'queue_time': 0.048400566, 'total_time': 0.064091854}, 'model_name': 'openai/gpt-oss-safeguard-20b', 'system_fingerprint': 'fp_b5ae46a825', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a04194-d4a3-7400-849d-3e262a379aec-0', tool_calls=[{'name': 'hospital_search', 'args': {'query': 'ICU visiting hours'}, 

In [147]:
from langchain_core.messages import HumanMessage,ToolMessage
query = "What are the ICU visiting hours?"




#### **Dynamic Tool Calling**

In [164]:
def tool_loop_execution(query):
    messages = [HumanMessage(query)]
    while True:

        response = llm_with_tools.invoke(messages)

        messages.append(response)

        if not response.tool_calls:
            break

        for tool_call in response.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            tool_id = tool_call['id']

            tool = tool_map[tool_name]
            result = tool.invoke(tool_args)
            print("Tool Name:",tool)
            messages.append(ToolMessage(content = str(result),tool_call_id = tool_id))
    return response

In [149]:
response = tool_loop_execution(query).content
print(response)


--- Document 1 ---
## General Ward

Visitors may visit patients in the General Ward during the scheduled visiting periods.

## Intensive Care Unit (ICU)

ICU visiting is restricted to the scheduled period. Visitors should follow instructions provided by hospital staff.

## Pediatrics

Pediatric department visiting hours are scheduled from 4:00 PM to 6:00 PM, Monday through Saturday.

## Outpatient Department

--- Document 2 ---
# ABC General Hospital — Visiting Hours

## General Visiting Schedule

| Area | Visiting Hours | Days |
|---|---|---|
| General Ward | 10:00 AM - 12:00 PM | Monday - Sunday |
| General Ward | 4:00 PM - 6:00 PM | Monday - Sunday |
| ICU | 2:00 PM - 3:00 PM | Monday - Sunday |
| Pediatrics | 4:00 PM - 6:00 PM | Monday - Saturday |
| Outpatient Department | 9:00 AM - 1:00 PM | Monday - Saturday |

## General Ward

Visitors may visit patients in the General Ward during the scheduled visiting periods. ## Intensive Care Unit (ICU)

ICU visiting is restricted to the s

In [150]:
patient_ids = patients_df["Id"].head(3).tolist()
patient_ids

['cc3eac4a-34a7-9a15-1522-8087135631b7',
 '8b9df7af-24b3-66ee-afeb-2734ebcbd33c',
 '65cbad6a-eb9d-420e-9bab-b4f9c6aab7db']

In [151]:
vectorstore._collection.count()

50

#### ***Hospital/RAG***

In [152]:
hospital_queries = [
    "What are the ICU visiting hours?",
    "What services are available in the Cardiology Department?",
    "Which department provides care for children and adolescents?",
    "What services does the hospital provide?",
    "Does the hospital provide pharmacy services?",
    "What diagnostic services are available at the hospital?",
    "What departments are available at ABC General Hospital?"
]
for query in hospital_queries:
    print("Query:", query)
    print("Response:", tool_loop_execution(query).content)
    print("-" * 60)

Query: What are the ICU visiting hours?

--- Document 1 ---
## General Ward

Visitors may visit patients in the General Ward during the scheduled visiting periods.

## Intensive Care Unit (ICU)

ICU visiting is restricted to the scheduled period. Visitors should follow instructions provided by hospital staff.

## Pediatrics

Pediatric department visiting hours are scheduled from 4:00 PM to 6:00 PM, Monday through Saturday.

## Outpatient Department

--- Document 2 ---
# ABC General Hospital — Visiting Hours

## General Visiting Schedule

| Area | Visiting Hours | Days |
|---|---|---|
| General Ward | 10:00 AM - 12:00 PM | Monday - Sunday |
| General Ward | 4:00 PM - 6:00 PM | Monday - Sunday |
| ICU | 2:00 PM - 3:00 PM | Monday - Sunday |
| Pediatrics | 4:00 PM - 6:00 PM | Monday - Saturday |
| Outpatient Department | 9:00 AM - 1:00 PM | Monday - Saturday |

## General Ward

Visitors may visit patients in the General Ward during the scheduled visiting periods. ## Intensive Care Unit (I

#### ***Patient Data***

In [153]:
patient_queries = [
    "What is the gender of patient cc3eac4a-34a7-9a15-1522-8087135631b7?",
    "What is the birth date of patient 8b9df7af-24b3-66ee-afeb-2734ebcbd33c?",
    "What city does patient 65cbad6a-eb9d-420e-9bab-b4f9c6aab7db live in?",
    "What is the healthcare coverage of patient 8b9df7af-24b3-66ee-afeb-2734ebcbd33c?",
    "Give me the complete information for patient cc3eac4a-34a7-9a15-1522-8087135631b7?"
]
for query in patient_queries:
    print("Query:", query)
    print("Response:", tool_loop_execution(query).content)
    print("-" * 60)

Query: What is the gender of patient cc3eac4a-34a7-9a15-1522-8087135631b7?
Tool Name: name='search_patient' description="Search the hospital's synthetic patient records using a patient ID\n     and return the matching patient information." args_schema=<class 'langchain_core.utils.pydantic.search_patient'> func=<function search_patient at 0x000001D2F1182D40>
Response: The patient’s gender is **male** (M).
------------------------------------------------------------
Query: What is the birth date of patient 8b9df7af-24b3-66ee-afeb-2734ebcbd33c?
Tool Name: name='search_patient' description="Search the hospital's synthetic patient records using a patient ID\n     and return the matching patient information." args_schema=<class 'langchain_core.utils.pydantic.search_patient'> func=<function search_patient at 0x000001D2F1182D40>
Response: The patient’s birth date is **February 11, 2001**.
------------------------------------------------------------
Query: What city does patient 65cbad6a-eb9d-4

#### ***Direct/No Tool***

In [154]:
direct_queries = [
    "What is LangChain?",
    "What is the difference between RAG and fine-tuning?",
    "What is an embedding?",
    "What is cosine similarity?"
]
for query in direct_queries:
    print("Query:", query)
    print("Response:", tool_loop_execution(query).content)
    print("-" * 60)

Query: What is LangChain?
Response: **LangChain** is an open‑source framework designed to help developers build applications that combine large language models (LLMs) with other tools, data sources, and external services. It abstracts away many of the low‑level details involved in creating “chain”‑based workflows where a language model can:

1. **Retrieve** information from external sources (e.g., databases, APIs, files).
2. **Process** that information using prompts or custom logic.
3. **Return** refined outputs or trigger subsequent actions.

Key concepts in LangChain:

- **Chains**: Sequential or conditional steps that involve LLM calls, function calls, or other operations.
- **Agents**: LLM-driven decision engines that can pick which tool to use based on a prompt.
- **Prompt Templates**: Reusable prompt structures that can be filled with dynamic data.
- **Memory**: Contextual state that can be stored and passed between steps.
- **Tool Integration**: Plug‑in interfaces for APIs, dat

#### ***Missing Information***

In [155]:
missing_queries = [
    "What is the cardiologist's phone number?",
    "What is the hospital's email address?",
    "Who is the head of the Cardiology Department?"
]
for query in missing_queries:
    print("Query:", query)
    print("Response:", tool_loop_execution(query).content)
    print("-" * 60)

Query: What is the cardiologist's phone number?

--- Document 1 ---
## Important Note

These visiting hours are fictional and created for RAG experimentation. They are not the schedule of a real hospital.

--- Document 2 ---
## Important Note

This is fictional hospital information created for a LangChain RAG project. It is not clinical guidance.

--- Document 3 ---
## Outpatient Department

The Outpatient Department operates from 9:00 AM to 1:00 PM, Monday through Saturday.

## Visitor Guidelines

- Visitors should follow hospital staff instructions.
- Visiting conditions may be changed for operational or safety reasons.
- Visitors should avoid disturbing patients or clinical activities.

## Important Note

--- Document 4 ---
## General Ward

Visitors may visit patients in the General Ward during the scheduled visiting periods.

## Intensive Care Unit (ICU)

ICU visiting is restricted to the scheduled period. Visitors should follow instructions provided by hospital staff.

## Pediatri

#### ***Mixed Query***

In [156]:
mixed_queries = [
    "What services are available in the Cardiology Department, and what is the gender of patient cc3eac4a-34a7-9a15-1522-8087135631b7?"
]
for query in mixed_queries:
    print("Query:", query)
    print("Response:", tool_loop_execution(query).content)
    print("-" * 60)

Query: What services are available in the Cardiology Department, and what is the gender of patient cc3eac4a-34a7-9a15-1522-8087135631b7?
Tool Name: name='search_patient' description="Search the hospital's synthetic patient records using a patient ID\n     and return the matching patient information." args_schema=<class 'langchain_core.utils.pydantic.search_patient'> func=<function search_patient at 0x000001D2F1182D40>

--- Document 1 ---
## Cardiology

The Cardiology Department focuses on evaluation and management of cardiovascular conditions.

## Neurology

The Neurology Department focuses on conditions involving the brain, spinal cord, nerves, and related neurological functions.

## Orthopedics

The Orthopedics Department focuses on musculoskeletal conditions involving bones, joints, muscles, ligaments, and related structures.

--- Document 2 ---
| Department | Main Area | Typical Services |
|---|---|---|
| Cardiology | Heart and cardiovascular system | Cardiac evaluation and cardiov

#### ***Evaluation Dataset***

In [206]:
test_queries = [
    "What services are available in the Cardiology Department?",
    "What diagnostic services are available at the hospital?",
    "What is the cardiologist's phone number?"
]

for query in test_queries:
    print("Query:", query)
    print("Response:", tool_loop_execution(query).content)
    print("-" * 60)

Query: What services are available in the Cardiology Department?
Tool Name: name='hospital_search' description='Search the hospital knowledge base and answer questions using information retrieved from the hospital documents.' args_schema=<class 'langchain_core.utils.pydantic.hospital_search'> func=<function hospital_search at 0x000001D2F1180400>
Response: **Cardiology Department – Available Services**

| Service | Brief Description |
|---------|-------------------|
| **Cardiac Evaluation & Cardiovascular Care** | Comprehensive assessment and ongoing management of all heart‑related conditions. |
| **Diagnostic Testing** | Includes ECG, Holter monitoring, stress testing, echocardiography, and cardiac imaging to evaluate heart structure and function. |
| **Interventional Cardiology** | Percutaneous coronary interventions (PCI), angioplasty, stent placement, and related catheter‑based procedures. |
| **Electrophysiology** | Arrhythmia evaluation and treatment, including pacemaker and defib